# Scraper: Raw Mastodon Posts

This section copies the Mastodon scraping logic into notebook cells so you can load event and instance configs, collect raw posts, and write them into a JSONL file without using the original CLI entry point.


In [5]:
import time
import datetime
import json
import os
from mastodon import Mastodon, MastodonNotFoundError


In [6]:
def load_config_files(instances_path, events_path):
    instances = []
    if os.path.exists(instances_path):
        with open(instances_path, 'r') as f:
            instances = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    else:
        print(f"[WARN] Instances file '{instances_path}' not found.")
        
    events = []
    if os.path.exists(events_path):
        with open(events_path, 'r') as f:
            events = json.load(f)
    else:
        print(f"[WARN] Events file '{events_path}' not found.")
            
    return instances, events


def save_to_file(data_list, filepath):
    if not data_list:
        return
    with open(filepath, 'a', encoding='utf-8') as f:
        for entry in data_list:
            json.dump(entry, f, default=str)
            f.write('\n')


In [7]:
def scrape_event(instance_url, event, existing_ids, window_days, output_file, access_token=None):
    target_date = datetime.datetime.strptime(event['date'], "%Y-%m-%d")
    start_date = target_date - datetime.timedelta(days=window_days)
    end_window = target_date + datetime.timedelta(days=window_days)
    max_id = int(end_window.timestamp() * 1000) << 16
    
    try:
        if access_token:
            client = Mastodon(api_base_url=instance_url, access_token=access_token, request_timeout=10)
        else:
            client = Mastodon(api_base_url=instance_url, request_timeout=10)
    except Exception as e:
        print(f"[ERROR] Connection failed: {e}")
        return 0

    total_scraped_for_event = 0
    max_pages = 50
    
    for tag in event.get('hashtags', []):
        current_max_id = max_id
        batch = []
        seen_ids = set()
        
        for _ in range(max_pages):
            try:
                timeline = client.timeline_hashtag(tag, local=True, limit=40, max_id=current_max_id)
                if not timeline:
                    break 
                
                current_max_id = timeline[-1]['id']
                
                for toot in timeline:
                    created_at = toot['created_at']
                    if created_at.tzinfo:
                        created_at = created_at.replace(tzinfo=None)
                    
                    if created_at < start_date:
                        continue
                    if toot['id'] in seen_ids:
                        continue
                    
                    unique_id = f"{instance_url}_{toot['id']}"
                    if unique_id in existing_ids:
                        continue
                    
                    seen_ids.add(toot['id'])
                    existing_ids.add(unique_id)
                    
                    doc = {
                        "id": unique_id,
                        "event": event['name'],
                        "event_date": event['date'],
                        "instance": instance_url,
                        "content": toot['content'],
                        "created_at": toot['created_at'],
                        "hashtag_searched": tag,
                        "url": toot['url']
                    }
                    batch.append(doc)
                
                last_ts = timeline[-1]['created_at']
                if last_ts.tzinfo:
                    last_ts = last_ts.replace(tzinfo=None)
                if last_ts < start_date:
                    break
                
                time.sleep(0.2)
            except (MastodonNotFoundError, Exception):
                break
            
        if batch:
            save_to_file(batch, output_file)
            total_scraped_for_event += len(batch)
    
    return total_scraped_for_event


Use the cell below to set file paths, the day window, and your optional access token. The helper loads existing IDs (to avoid duplicates) and loops through every event/instance combination.


In [8]:
scrape_instances_path = "instances.txt"
scrape_events_path = "ai_events.json"
scrape_output_path = "mastodon_raw.json"
scrape_window_days = 7
scrape_token_file = None  # e.g., "token.txt" or None if not needed

instances, events = load_config_files(scrape_instances_path, scrape_events_path)
existing_ids = set()

if os.path.exists(scrape_output_path):
    with open(scrape_output_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                existing_ids.add(json.loads(line).get('id'))

access_token = None
if scrape_token_file and os.path.exists(scrape_token_file):
    with open(scrape_token_file, 'r') as f:
        access_token = f.read().strip()

for event in events:
    for instance in instances:
        new_posts = scrape_event(instance, event, existing_ids, scrape_window_days, scrape_output_path, access_token)
        if new_posts:
            print(f"Archived {new_posts} posts for {event['name']} on {instance}")


Archived 1496 posts for ChatGPT Release on https://mastodon.social
Archived 313 posts for ChatGPT Release on https://fosstodon.org
Archived 334 posts for ChatGPT Release on https://sigmoid.social
Archived 182 posts for ChatGPT Release on https://mas.to
Archived 116 posts for ChatGPT Release on https://mastodon.world
Archived 345 posts for ChatGPT Release on https://infosec.exchange
Archived 196 posts for ChatGPT Release on https://mstdn.social
Archived 125 posts for ChatGPT Release on https://universeodon.com
Archived 341 posts for ChatGPT Release on https://mastodon.online
Archived 36 posts for ChatGPT Release on https://social.vivaldi.net
Archived 15 posts for ChatGPT Release on https://mstdn.jp
Archived 44 posts for ChatGPT Release on https://mastodon.cloud
Archived 29 posts for ChatGPT Release on https://mastodon.art
Archived 104 posts for ChatGPT Release on https://techhub.social
Archived 10 posts for ChatGPT Release on https://ruby.social
Archived 17 posts for ChatGPT Release on 

# Sentiment & Topic Processing

This stage mirrors `sandy.py` and enriches the scraped JSONL file with TextBlob, VADER, optional emotion scores, and topic modeling. Cells stay modular so you can run the analysis incrementally without CLI arguments.


In [9]:
import re
import numpy as np
from textblob import TextBlob

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader_analyzer = SentimentIntensityAnalyzer()
    VADER_AVAILABLE = True
except ImportError:
    VADER_AVAILABLE = False

try:
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.decomposition import LatentDirichletAllocation
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

try:
    import text2emotion as te
    EMOTION_AVAILABLE = True
except ImportError:
    EMOTION_AVAILABLE = False


ModuleNotFoundError: No module named 'textblob'

In [ ]:
def clean_text_content(text):
    clean = text.replace("<br>", " ").replace("<p>", "").replace("</p>", "")
    clean = re.sub(r'http\S+', '', clean)
    return clean


def calculate_metrics(content):
    text = clean_text_content(content)
    blob = TextBlob(text)
    tb_polarity = blob.sentiment.polarity
    tb_subjectivity = blob.sentiment.subjectivity
    
    vader_score = 0.0
    if VADER_AVAILABLE:
        vader_score = vader_analyzer.polarity_scores(text)['compound']
        
    emotions = {}
    if EMOTION_AVAILABLE:
        try:
            emotions = te.get_emotion(text)
        except Exception:
            emotions = {'Happy': 0, 'Angry': 0, 'Surprise': 0, 'Sad': 0, 'Fear': 0}
            
    return tb_polarity, tb_subjectivity, vader_score, emotions


def classify_sentiment(polarity):
    if polarity > 0.05:
        return 'Positive'
    if polarity < -0.05:
        return 'Negative'
    return 'Neutral'


In [ ]:
def apply_topic_modeling(data_list):
    if not SKLEARN_AVAILABLE:
        return data_list
    
    events = {}
    for doc in data_list:
        evt = doc.get('event', 'Unknown')
        events.setdefault(evt, []).append(doc)
    
    for docs in events.values():
        corpus = [d.get('content', '') for d in docs]
        if len(corpus) < 5:
            continue
        
        tf_vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
        try:
            tf = tf_vectorizer.fit_transform(corpus)
            feature_names = tf_vectorizer.get_feature_names_out()
        except ValueError:
            continue
        
        n_topics = 3
        lda = LatentDirichletAllocation(n_components=n_topics, max_iter=5, learning_method='online', random_state=0)
        lda.fit(tf)
        topic_distributions = lda.transform(tf)
        
        topic_keywords = {}
        for topic_idx, topic in enumerate(lda.components_):
            top_indices = topic.argsort()[:-4:-1]
            keywords = [feature_names[i] for i in top_indices]
            topic_keywords[topic_idx] = ", ".join(keywords)
        
        for i, doc in enumerate(docs):
            dominant_topic_idx = int(topic_distributions[i].argmax())
            doc['topic_id'] = dominant_topic_idx
            doc['topic_keywords'] = topic_keywords[dominant_topic_idx]
    
    return data_list


In [ ]:
def process_database(input_file, output_file, skip_emotion=False):
    if not os.path.exists(input_file):
        print(f"[ERROR] Input file '{input_file}' not found.")
        return
    
    raw_data = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                raw_data.append(json.loads(line))
    
    total_lines = len(raw_data)
    print(f"Loaded {total_lines} records for enrichment.")
    
    processed_data = []
    for record in raw_data:
        try:
            content = record.get('content', '')
            pol, subj, vader, emotions = calculate_metrics(content)
            record['polarity'] = pol
            record['subjectivity'] = subj
            record['vader_score'] = vader
            record['sentiment_class'] = classify_sentiment(pol)
            if emotions and not skip_emotion:
                for emo, val in emotions.items():
                    record[f'emotion_{emo.lower()}'] = val
            processed_data.append(record)
        except Exception:
            continue
    
    if SKLEARN_AVAILABLE:
        processed_data = apply_topic_modeling(processed_data)
    else:
        print("[WARN] scikit-learn missing. Topic modeling skipped.")
    
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for entry in processed_data:
            json.dump(entry, outfile, default=str)
            outfile.write('\n')
    
    print(f"Saved {len(processed_data)} enriched records to '{output_file}'.")


Configure the enrichment step by pointing to the raw JSONL file generated earlier, choosing an output file, and deciding whether to include the optional emotion model.


In [ ]:
analysis_input_file = "mastodon_raw.json"
analysis_output_file = "mastodon_analyzed.json"
skip_emotion = False  # Set True to skip emotion extraction for speed

process_database(analysis_input_file, analysis_output_file, skip_emotion=skip_emotion)


# Visualization & Reporting

The remaining cells reproduce `analyzer.py`: load the enriched dataset, prepare helper utilities, and render each report figure in its own notebook cell with a short explanation.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datetime import datetime
from math import pi, ceil

try:
    import networkx as nx
    NETWORKX_AVAILABLE = True
except ImportError:
    NETWORKX_AVAILABLE = False

STOPWORDS = set([
    'the', 'and', 'to', 'of', 'a', 'in', 'is', 'for', 'on', 'that', 'it', 'with', 'as', 'are', 
    'this', 'was', 'by', 'an', 'be', 'or', 'at', 'from', 'not', 'have', 'has', 'but', 'can', 
    'more', 'about', 'we', 'my', 'they', 'what', 'so', 'like', 'just', 'https', 'http', 'com', 
    'www', 'mastodon', 'social', 'content', 'html', 'href', 'rel', 'nofollow', 'target', 'blank',
    'span', 'class', 'br', 'p', 'div', 'label', 'translate', 'search', 'status', 'card', 'if', 
    'you', 'me', 'your', 'will', 'one', 'all', 'do', 'no', 'up', 'out', 'there', 'get', 'how',
    'when', 'some', 'time', 'now', 'only', 'new', 'amp', 'gt', 'lt', 'quot', 'people', 'ai'
])


def load_data(filepath):
    if not os.path.exists(filepath):
        print(f"[ERROR] File '{filepath}' not found.")
        return pd.DataFrame()

    data = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    except Exception as e:
        print(f"[ERROR] Corrupt data: {e}")
        return pd.DataFrame()
    
    df = pd.DataFrame(data)
    
    if not df.empty:
        df['created_at'] = pd.to_datetime(df['created_at'], format='mixed', utc=True)
        df['event_date'] = pd.to_datetime(df['event_date'], format='mixed', utc=True)
        df['label'] = df['event'] + "\n(" + df['event_date'].dt.strftime('%Y-%m-%d') + ")"
    
    print(f"Loaded {len(df)} analyzed records.")
    return df


def extract_keywords(text_series, top_n=10):
    all_words = []
    for text in text_series:
        if not isinstance(text, str):
            continue
        clean = re.sub(r'<[^>]+>', '', text).lower()
        clean = re.sub(r'[^\w\s]', '', clean)
        words = clean.split()
        all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 3])
    return Counter(all_words).most_common(top_n)


def extract_emojis(text_series, top_n=10):
    emoji_pattern = re.compile(r'[\U0001F000-\U0001F9FF]|[\u2700-\u27BF]|[\u2600-\u26FF]')
    all_emojis = []
    for text in text_series:
        if not isinstance(text, str):
            continue
        all_emojis.extend(emoji_pattern.findall(text))
    return Counter(all_emojis).most_common(top_n)


def extract_mentions(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'@([\w\d_]+)', text)


Set the analyzed file path, load it into a DataFrame, and prepare helper structures shared by the visualization cells.


In [ ]:
analyzed_input_path = "mastodon_analyzed.json"
figure_output_dir = "figures"
os.makedirs(figure_output_dir, exist_ok=True)

df = load_data(analyzed_input_path)

if not df.empty:
    unique_events = df[['label', 'event_date', 'event']].drop_duplicates().sort_values('event_date')
    chronological_labels = unique_events['label'].tolist()
    event_names = unique_events['event'].tolist()
else:
    unique_events = pd.DataFrame(columns=['label', 'event_date', 'event'])
    chronological_labels = []
    event_names = []


### Timeline sentiment and reaction split
This view shows the average polarity per event and how positive/neutral/negative posts stack up for each launch window.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    plt.figure(figsize=(16, 10))

    plt.subplot(2, 1, 1)
    pivot_avg = df.groupby('label')['polarity'].mean().reindex(chronological_labels)
    colors = ['#ff4d4d' if x < 0 else '#4dff88' for x in pivot_avg.values]
    pivot_avg.plot(kind='bar', color=colors, edgecolor='black', alpha=0.8)
    plt.title('Average Sentiment Polarity per Event', fontsize=14, fontweight='bold')
    plt.axhline(0, color='black', linewidth=1)
    plt.xticks([])
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    plt.subplot(2, 1, 2)
    sentiment_counts = df.groupby(['label', 'sentiment_class']).size().unstack(fill_value=0)
    sentiment_counts = sentiment_counts.reindex(chronological_labels)
    sentiment_pct = sentiment_counts.div(sentiment_counts.sum(axis=1), axis=0) * 100
    for col in ['Negative', 'Neutral', 'Positive']:
        if col not in sentiment_pct:
            sentiment_pct[col] = 0
    sentiment_pct[['Negative', 'Neutral', 'Positive']].plot(
        kind='bar', stacked=True, color=['#ff6666', '#e0e0e0', '#66cc66'],
        ax=plt.gca(), edgecolor='black', width=0.8
    )
    plt.title('Community Reaction Split', fontsize=14, fontweight='bold')
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "01_timeline_sentiment.png"))
    plt.show()


### Share of voice pie chart
See how much conversation each event captured relative to the others.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    plt.figure(figsize=(10, 8))
    event_counts = df['event'].value_counts()
    if len(event_counts) > 10:
        top_events = event_counts.head(9)
        other_count = event_counts.iloc[9:].sum()
        top_events['Other'] = other_count
        event_counts = top_events
    plt.pie(event_counts, labels=event_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
    plt.title('Share of Voice: Dominant Events', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "02_share_of_voice.png"))
    plt.show()


### Volume spikes around launch day
This line chart tracks how post volume rises and falls relative to each event date.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    plt.figure(figsize=(12, 6))
    vol_df = df.copy()
    vol_df['days_relative'] = (vol_df['created_at'] - vol_df['event_date']).dt.days
    vol_df = vol_df[(vol_df['days_relative'] >= -2) & (vol_df['days_relative'] <= 7)]
    pivot_vol = vol_df.groupby(['days_relative', 'event']).size().unstack(fill_value=0)
    pivot_vol.plot(kind='line', linewidth=2, marker='o', figsize=(12, 6))
    plt.title('Hype Cycle: Discussion Volume Relative to Launch Day', fontsize=14, fontweight='bold')
    plt.xlabel('Days Relative to Event (0 = Launch Day)', fontsize=12)
    plt.ylabel('Post Volume', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "03_volume_spikes.png"))
    plt.show()


### Weekend effect heatmap
Highlights which days of the week and hours of the day drive the most posting activity.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    plt.figure(figsize=(12, 6))
    time_df = df.copy()
    time_df['day_name'] = time_df['created_at'].dt.day_name()
    time_df['hour'] = time_df['created_at'].dt.hour
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    time_df['day_name'] = pd.Categorical(time_df['day_name'], categories=days_order, ordered=True)
    pivot_time = time_df.groupby(['day_name', 'hour'], observed=False).size().unstack(fill_value=0)
    sns.heatmap(pivot_time, cmap="YlGnBu", cbar_kws={'label': 'Post Volume'})
    plt.title('Weekend Effect: Activity Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "04_weekend_effect.png"))
    plt.show()


### Sentiment velocity
Tracks how the average polarity changes in the days immediately before and after each event.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    plt.figure(figsize=(12, 6))
    vel_df = df.copy()
    vel_df['days_relative'] = (vel_df['created_at'] - vel_df['event_date']).dt.days
    vel_df = vel_df[(vel_df['days_relative'] >= -2) & (vel_df['days_relative'] <= 7)]
    pivot_velocity = vel_df.groupby(['days_relative', 'event'])['polarity'].mean().unstack()
    pivot_velocity.plot(linewidth=2, figsize=(12, 6), alpha=0.8)
    plt.title('Sentiment Velocity: Post-Launch Shifts', fontsize=14, fontweight='bold')
    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.xlabel('Days Relative to Event', fontsize=12)
    plt.ylabel('Average Polarity', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "05_sentiment_velocity.png"))
    plt.show()


### Fact vs. opinion scatter
Plots average polarity vs. subjectivity for each event to see which launches sparked opinionated takes.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    plt.figure(figsize=(10, 8))
    scatter_data = df.groupby('event')[['polarity', 'subjectivity']].mean().reset_index()
    sns.scatterplot(data=scatter_data, x='polarity', y='subjectivity', s=200, hue='event', style='event')
    for i in range(scatter_data.shape[0]):
        plt.text(scatter_data.polarity[i] + 0.01, scatter_data.subjectivity[i], scatter_data.event[i], fontsize=9)
    plt.axvline(0, color='gray', linestyle='--')
    plt.axhline(0.5, color='gray', linestyle='--')
    plt.title('Fact vs. Opinion Matrix', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "06_hot_take_matrix.png"))
    plt.show()


### Instance landscape
Charts subjectivity vs. polarity for instances with enough posts to spot outliers.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    inst_stats = df.groupby('instance').agg({'polarity': 'mean', 'subjectivity': 'mean', 'id': 'count'}).reset_index()
    inst_stats = inst_stats[inst_stats['id'] > 10]
    if inst_stats.empty:
        print("No instances have more than 10 posts; skipping plot.")
    else:
        plt.figure(figsize=(12, 8))
        sns.scatterplot(data=inst_stats, x='subjectivity', y='polarity', size='id', sizes=(50, 1000), alpha=0.6, color='purple', legend=False)
        top_insts = inst_stats.nlargest(5, 'id')
        for _, row in top_insts.iterrows():
            plt.text(row['subjectivity'], row['polarity'], row['instance'].split('//')[-1], fontsize=10, fontweight='bold')
        plt.title('Instance Landscape', fontsize=14, fontweight='bold')
        plt.xlabel('Average Subjectivity')
        plt.ylabel('Average Polarity')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(figure_output_dir, "07_instance_density.png"))
        plt.show()


### Post length by sentiment
Shows whether positive, neutral, or negative reactions tend to be longer.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    plt.figure(figsize=(10, 6))
    df['length'] = df['content'].astype(str).apply(len)
    sns.histplot(data=df, x='length', hue='sentiment_class', element="step", stat="density", common_norm=False)
    plt.xlim(0, 1000)
    plt.title('Post Length by Sentiment', fontsize=14, fontweight='bold')
    plt.xlabel('Characters per Post')
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "08_post_length.png"))
    plt.show()


### Emoji usage heatmap
Counts the most common emojis per event to catch mood cues that sentiment scores may miss.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    all_emojis = extract_emojis(df['content'], top_n=50)
    if not all_emojis:
        print("No emojis found in the dataset.")
    else:
        top_10_emojis = [e[0] for e in all_emojis[:10]]
        emoji_matrix = []
        event_labels = []
        for evt in unique_events['event'].unique():
            evt_text = " ".join(df[df['event'] == evt]['content'].astype(str))
            counts = [evt_text.count(emo) for emo in top_10_emojis]
            emoji_matrix.append(counts)
            event_labels.append(evt)
        if emoji_matrix:
            plt.figure(figsize=(10, 8))
            sns.heatmap(emoji_matrix, annot=True, fmt="d", xticklabels=top_10_emojis, yticklabels=event_labels, cmap="Oranges")
            plt.title('Emoji Usage Heatmap', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(os.path.join(figure_output_dir, "09_emoji_heatmap.png"))
            plt.show()
        else:
            print("No events contained emoji data.")


### Keyword drivers per event
Breaks out the most common positive and negative words for each event (minimum five posts).


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    created_any = False
    for evt in unique_events['event'].unique():
        evt_df = df[df['event'] == evt]
        if len(evt_df) < 5:
            continue
        pos_text = evt_df[evt_df['sentiment_class'] == 'Positive']['content']
        neg_text = evt_df[evt_df['sentiment_class'] == 'Negative']['content']
        pos_keywords = extract_keywords(pos_text)
        neg_keywords = extract_keywords(neg_text)
        if not pos_keywords and not neg_keywords:
            continue
        created_any = True
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
        fig.suptitle(f'Context Analysis: {evt}', fontsize=16, fontweight='bold')
        if pos_keywords:
            words, counts = zip(*pos_keywords)
            ax1.barh(words, counts, color='#66cc66', edgecolor='black')
            ax1.set_title('Positive Drivers')
            ax1.invert_yaxis()
        else:
            ax1.set_visible(False)
        if neg_keywords:
            words, counts = zip(*neg_keywords)
            ax2.barh(words, counts, color='#ff6666', edgecolor='black')
            ax2.set_title('Negative Drivers')
            ax2.invert_yaxis()
        else:
            ax2.set_visible(False)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        safe_name = "".join(x for x in evt if x.isalnum())
        plt.savefig(os.path.join(figure_output_dir, f"10_keywords_{safe_name}.png"))
        plt.show()
    if not created_any:
        print("Not enough keyword data to render charts.")


### Before vs. after sentiment shift
Shows how sentiment mix and volume change when comparing posts before and after each event date.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    ba_df = df.copy()
    ba_df['period'] = np.where(ba_df['created_at'] < ba_df['event_date'], 'Before', 'After')
    ba_counts = ba_df.groupby(['label', 'period', 'sentiment_class']).size().unstack(fill_value=0)
    for col in ['Negative', 'Neutral', 'Positive']:
        if col not in ba_counts:
            ba_counts[col] = 0

    def get_sort_key(index_tuple):
        lbl, per = index_tuple
        try:
            date_str = lbl.split('(')[-1].replace(')', '')
            dt = datetime.strptime(date_str, '%Y-%m-%d')
        except Exception:
            dt = datetime.min
        period_rank = 0 if per == 'Before' else 1
        return (dt, lbl, period_rank)

    sorted_idx = sorted(ba_counts.index, key=get_sort_key)
    ba_counts = ba_counts.reindex(sorted_idx)
    plt.figure(figsize=(18, 8))
    ba_counts[['Negative', 'Neutral', 'Positive']].plot(
        kind='bar', stacked=True, color=['#ff6666', '#e0e0e0', '#66cc66'],
        edgecolor='black', width=0.8, figsize=(18, 8)
    )
    plt.title('Volume & Sentiment Shift: Before vs. After', fontsize=16, fontweight='bold')
    plt.ylabel('Number of Posts', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "11_before_after_shift.png"))
    plt.show()


### Emotion radar charts
Visualize text2emotion outputs (if available) across the core emotion dimensions for each event.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    emotion_cols = [c for c in df.columns if c.startswith('emotion_')]
    if not emotion_cols:
        print("No emotion columns present; skip radar charts.")
    else:
        emo_means = df.groupby('event')[emotion_cols].mean()
        valid_events = [e for e in event_names if e in emo_means.index]
        if not valid_events:
            print("No events contain emotion data.")
        else:
            categories = [c.replace('emotion_', '').capitalize() for c in emotion_cols]
            N = len(categories)
            angles = [n / float(N) * 2 * pi for n in range(N)]
            angles += [angles[0]]
            cols = 3
            rows = ceil(len(valid_events) / cols)
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5), subplot_kw={'projection': 'polar'})
            axes = axes.flatten()
            for idx, evt in enumerate(valid_events):
                ax = axes[idx]
                values = emo_means.loc[evt].values.tolist()
                values += [values[0]]
                ax.plot(angles, values, linewidth=2, linestyle='solid', color='blue')
                ax.fill(angles, values, color='blue', alpha=0.25)
                ax.set_xticks(angles[:-1])
                ax.set_xticklabels(categories, fontsize=8)
                ax.set_title(evt, size=12, fontweight='bold', y=1.1)
                ax.grid(True)
            for i in range(len(valid_events), len(axes)):
                axes[i].axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(figure_output_dir, "12_emotion_radar.png"))
            plt.show()


### Topic diversity
Summaries the most frequent inferred topics per event when LDA output is available.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    if 'topic_keywords' not in df.columns:
        print("No topic modeling data present.")
    else:
        valid_events = [e for e in event_names if not df[df['event'] == e]['topic_keywords'].isna().all()]
        if not valid_events:
            print("No events contain topic assignments.")
        else:
            cols = 2
            rows = ceil(len(valid_events) / cols)
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 8, rows * 4))
            axes = axes.flatten()
            for idx, evt in enumerate(valid_events):
                ax = axes[idx]
                evt_data = df[df['event'] == evt]
                topic_counts = evt_data['topic_keywords'].value_counts().head(5)
                labels = [", ".join(k.split(', ')[:3]) for k in topic_counts.index]
                ax.barh(labels, topic_counts.values, color='teal', edgecolor='black', alpha=0.7)
                ax.set_title(evt, fontsize=12, fontweight='bold')
                ax.invert_yaxis()
            for i in range(len(valid_events), len(axes)):
                axes[i].axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(figure_output_dir, "13_topic_distribution.png"))
            plt.show()


### Influencer mentions
Either draws a simple mention bar chart or a small network (if NetworkX is installed).


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    mentions = []
    for text in df['content']:
        mentions.extend(extract_mentions(str(text)))
    if not mentions:
        print("No mentions found in the dataset.")
    else:
        top_mentions = Counter(mentions).most_common(20)
        users, counts = zip(*top_mentions)
        if NETWORKX_AVAILABLE:
            G = nx.Graph()
            for user, count in top_mentions:
                G.add_node(user, size=count)
                G.add_edge("Community", user, weight=count)
            plt.figure(figsize=(12, 12))
            pos = nx.spring_layout(G, k=0.5)
            sizes = [G.nodes[n].get('size', 10) * 50 for n in G.nodes]
            nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color='skyblue', alpha=0.7)
            nx.draw_networkx_edges(G, pos, width=1, alpha=0.3)
            nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
            plt.title('Top Mentioned Accounts (Influencer Map)', fontsize=14, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(figure_output_dir, "14_influencer_network.png"))
            plt.show()
        else:
            plt.figure(figsize=(12, 8))
            plt.barh(users, counts, color='skyblue', edgecolor='black')
            plt.title('Top Mentioned Accounts', fontsize=14, fontweight='bold')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.savefig(os.path.join(figure_output_dir, "14_influencer_bar.png"))
            plt.show()


### Posting hour distribution
A 24-hour histogram (UTC) that hints at dominant regions engaging with the topic.


In [ ]:
if df.empty:
    print("No analyzed data available for plotting.")
else:
    hours = df['created_at'].dt.hour
    plt.figure(figsize=(12, 6))
    sns.histplot(hours, bins=24, kde=True, color='teal', edgecolor='black')
    plt.title('Global Activity: Posting Hour Distribution (UTC)', fontsize=14, fontweight='bold')
    plt.xlabel('Hour of Day (UTC)', fontsize=12)
    plt.xticks(range(0, 25, 2))
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.axvspan(8, 17, color='yellow', alpha=0.1, label='EU/Africa Working Hours')
    plt.axvspan(14, 23, color='blue', alpha=0.1, label='Americas Working Hours')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(figure_output_dir, "15_timezone_inference.png"))
    plt.show()
